# Unit 6: Text-to-Speech (TTS) - Hands-on Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 6](https://huggingface.co/learn/audio-course/chapter6)

## Certificate Exercise Requirements

- Fine-tune **SpeechT5** on a TTS dataset of your choosing
- Push the model to the Hub tagged as `text-to-speech`
- No specific metric threshold -- focus on practice and learning
- Keep training data to ~10-15 hours for free Colab GPU

## Status: TODO

## Setup

In [ ]:
!pip install -q transformers datasets soundfile speechbrain accelerate librosa

from huggingface_hub import notebook_login
notebook_login()

## 1. Load a TTS Dataset

Good options:
- `facebook/voxpopuli` (multiple languages, parliament speeches)
- `mozilla-foundation/common_voice_13_0` (crowd-sourced, many languages)

We'll use VoxPopuli Dutch as the course example uses, but you can pick any language.

In [ ]:
from datasets import load_dataset, Audio

# Choose your language/dataset here
# Example: Dutch from VoxPopuli
dataset = load_dataset("facebook/voxpopuli", "nl", split="train", streaming=True, trust_remote_code=True)

# Take a manageable subset (~10 hours)
# Adjust the number based on average audio duration in your chosen dataset
dataset_size = 5000  # Adjust as needed

# Preview
sample = next(iter(dataset))
print(f"Keys: {sample.keys()}")
print(f"Text: {sample.get('raw_text', sample.get('sentence', 'N/A'))}")
print(f"Sampling rate: {sample['audio']['sampling_rate']}")

In [ ]:
# Load non-streaming version with limited examples
dataset = load_dataset("facebook/voxpopuli", "nl", split=f"train[:{dataset_size}]", trust_remote_code=True)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

## 2. Load SpeechT5 Components

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")

# SpeechT5 tokenizer only handles English characters by default
# For other languages, we may need to handle special characters
tokenizer = processor.tokenizer
print(f"Vocab size: {tokenizer.vocab_size}")

## 3. Prepare Speaker Embeddings

SpeechT5 needs speaker embeddings (x-vectors) to condition the output voice.

In [ ]:
from speechbrain.inference.classifiers import EncoderClassifier
import torch

spk_model_name = "speechbrain/spkrec-xvect-voxceleb"
speaker_model = EncoderClassifier.from_hparams(
    source=spk_model_name,
    run_opts={"device": "cpu"},
)

def create_speaker_embedding(waveform):
    with torch.no_grad():
        speaker_embeddings = speaker_model.encode_batch(torch.tensor(waveform).unsqueeze(0))
        speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
        speaker_embeddings = speaker_embeddings.squeeze()
    return speaker_embeddings

## 4. Preprocess the Dataset

In [ ]:
# Resample to 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]
    text = example.get("raw_text", example.get("sentence", ""))
    
    # Tokenize text
    example["input_ids"] = processor.tokenizer(text).input_ids
    
    # Compute log-mel spectrogram (target)
    example["labels"] = processor.feature_extractor(
        audio_target=audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    
    # Compute speaker embedding
    example["speaker_embeddings"] = create_speaker_embedding(audio["array"]).numpy()
    
    return example

# Filter out long examples (SpeechT5 limited to ~600 spectrogram frames)
def is_not_too_long(example):
    audio_length = len(example["audio"]["array"]) / example["audio"]["sampling_rate"]
    return audio_length < 10.0  # Keep under 10 seconds

dataset = dataset.filter(is_not_too_long)
print(f"After filtering: {dataset}")

dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, num_proc=1)

## 5. Data Collator

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        label_features = [{"input_values": feature["labels"]} for feature in features]
        speaker_features = [feature["speaker_embeddings"] for feature in features]

        batch = processor.pad(input_ids=input_ids, labels=label_features, return_tensors="pt")

        # Replace padding with -100
        batch["labels"] = batch["labels"].masked_fill(
            batch.decoder_attention_mask.unsqueeze(-1).ne(1), -100.0
        )

        # Not used during fine-tuning
        del batch["decoder_attention_mask"]

        # Round down target lengths to multiple of reduction factor
        if model.config.reduction_factor > 1:
            target_lengths = torch.tensor([
                len(feature["labels"]) for feature in features
            ])
            target_lengths = target_lengths.new([
                length - length % model.config.reduction_factor for length in target_lengths
            ])
            max_length = max(target_lengths)
            batch["labels"] = batch["labels"][:, :max_length]

        batch["speaker_embeddings"] = torch.tensor(speaker_features)
        return batch

data_collator = TTSDataCollatorWithPadding(processor=processor)

## 6. Train

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="speecht5-finetuned-tts",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=2,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    load_best_model_at_end=True,
    greater_is_better=False,
    label_names=["labels"],
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=processor,
)

trainer.train()

## 7. Test the Model

In [ ]:
import IPython.display as ipd
from transformers import SpeechT5HifiGan

# Load vocoder to convert spectrogram to waveform
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

# Use a speaker embedding from the dataset
example = dataset["test"][0]
speaker_embeddings = torch.tensor(example["speaker_embeddings"]).unsqueeze(0)

# Generate speech
text = "Hello, this is a test of the fine-tuned text to speech model."
inputs = processor(text=text, return_tensors="pt")

with torch.no_grad():
    speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

ipd.Audio(speech.numpy(), rate=16000)

## 8. Push to Hub (Required for Certificate)

In [ ]:
kwargs = {
    "dataset_tags": "facebook/voxpopuli",
    "dataset": "VoxPopuli",
    "finetuned_from": "microsoft/speecht5_tts",
    "tasks": "text-to-speech",
}

trainer.push_to_hub(**kwargs)